### Building a simple income regression model and a loan classification model.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import scipy.stats as stats
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import r2_score, root_mean_squared_error,roc_auc_score, classification_report, confusion_matrix, RocCurveDisplay
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE

In [ ]:
# Import the dataset
df = pd.read_csv("../data_files/loan_data.csv")

In [ ]:
# Introspect the dataset
df.head()

In [ ]:
# Introspect the attributes and datatypes
df.info()

### DATA CLEANING

In [ ]:
# Remove unrealistic datapoints
df = df[(df["person_age"] <= 80) & (df["person_emp_exp"] <= df["person_age"])]

### EXPLORATORY ANALYSIS TO UNDERSTAND THE DATA.

In [ ]:
# Check for missing values
df.isna().sum()

In [ ]:
# Distribution of target variable (income)
plt.hist(df["person_income"], bins=30, color="lightgreen", edgecolor="black")
plt.title("Distribution of Income")
plt.xlabel("Income")
plt.ylabel("Frequency")
plt.show()

In [ ]:
# Distribution of credit score
plt.hist(df["credit_score"], bins=30, color="lightgreen", edgecolor="black")
plt.title("Distribution of Credit Score")
plt.xlabel("Credit Score")
plt.ylabel("Frequency")
plt.show()

In [ ]:
# Plot loan distribution status
df_plot = df.copy()
df_plot["loan_status"] = df_plot["loan_status"].map({0: "Default", 1: "Paid"})

sns.countplot(x="loan_status", data=df_plot, edgecolor="black", palette="Set2", order=["Paid", "Default"])
plt.title("Distribution of Loan Status")
plt.xlabel("Loan Status")
plt.ylabel("Frequency")
plt.show()

In [ ]:
# Boxplot showing distribution of education and income
sns.boxplot(x="person_education", y="person_income", data=df, color="lightgreen")
plt.title("Income by Education Level")
plt.xlabel("Education")
plt.ylabel("Income (USD)")
plt.xticks(rotation=45)
plt.show()

### DATA PREPROCESSING APPLIED TO SPLIT DATA TO AVOID DATA LEAKAGE


In [ ]:
# Deterministic ordinal encoding for education level
df["person_education"] = df["person_education"].map({
    "High School": 1, "Associate": 2, "Bachelor": 3, "Master": 4, "Doctorate": 5
})

In [ ]:
# Drop columns that would cause leakage:
# loan_percent_income  = loan_amnt / person_income — arithmetically encodes the target.
#                        A model can back-calculate income from it. Target leakage.
# loan_status          = whether the borrower eventually defaulted — a post-repayment outcome.
#                        Unknown at the time you would want to predict income.
#                        Keeping it would mean the model learns from information it cannot
#                        have in production. Temporal / downstream leakage.

cols_to_drop = ["person_income", "loan_percent_income", "loan_status"]

X = df.drop(columns=cols_to_drop).copy()
Y = df["person_income"]

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.3, random_state=42)

In [ ]:
# Clip data after split to avoid leakage
lower = Y_train.quantile(0.01)
upper = Y_train.quantile(0.99)

Y_train = Y_train.clip(lower=lower, upper=upper)
Y_test  = Y_test.clip(lower=lower, upper=upper)

In [ ]:
# Encode categorical variables with LabelEncoder
cols_to_encode = ["person_gender", "person_home_ownership", "loan_intent",
                  "previous_loan_defaults_on_file"]

le_encoders = {}
for col in cols_to_encode:
    le = LabelEncoder()
    X_train[col] = le.fit_transform(X_train[col])
    X_test[col]  = le.transform(X_test[col])
    le_encoders[col] = le


In [ ]:
# selecting model
model_lm = LinearRegression()

# train model
model_lm.fit(X_train, Y_train)

In [ ]:
# Get predictions and residuals
y_pred    = model_lm.predict(X_test)
residuals = Y_test - y_pred

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Residuals vs Fitted — checks Linearity
axes[0, 0].scatter(y_pred, residuals, alpha=0.5, color="lightgreen")
axes[0, 0].axhline(0, color="red", linestyle="--")
axes[0, 0].set_title("Residuals vs Fitted")
axes[0, 0].set_xlabel("Fitted Values")
axes[0, 0].set_ylabel("Residuals")

# Q-Q Plot — checks Normality
stats.probplot(residuals, plot=axes[0, 1])
axes[0, 1].set_title("Normal Q-Q")

# Scale-Location — checks Homoscedasticity
axes[1, 0].scatter(y_pred, np.sqrt(np.abs(residuals)), alpha=0.5, color="lightgreen")
axes[1, 0].axhline(200, color="red", linestyle="--")
axes[1, 0].set_title("Scale-Location")
axes[1, 0].set_xlabel("Fitted Values")
axes[1, 0].set_ylabel("\u221a|Residuals|")

# Residuals histogram — checks Normality of errors
axes[1, 1].hist(residuals, bins=30, color="lightgreen", edgecolor="black")
axes[1, 1].set_title("Distribution of Residuals")
axes[1, 1].set_xlabel("Residuals")
axes[1, 1].set_ylabel("Frequency")

plt.tight_layout()
plt.show()

In [ ]:
# Log transformation
X_train_log = X_train.copy()
X_test_log  = X_test.copy()
Y_train_log = np.log(Y_train)
Y_test_log  = np.log(Y_test)

In [ ]:
# Train the log model
model_lm_log = LinearRegression()
model_lm_log.fit(X_train_log, Y_train_log)

In [ ]:
y_pred_log    = model_lm_log.predict(X_test_log)
residuals_log = Y_test_log - y_pred_log

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Residuals vs Fitted
axes[0, 0].scatter(y_pred_log, residuals_log, alpha=0.5, color="steelblue")
axes[0, 0].axhline(0, color="red", linestyle="--")
axes[0, 0].set_title("Residuals vs Fitted (log model)")
axes[0, 0].set_xlabel("Fitted Values (log scale)")
axes[0, 0].set_ylabel("Residuals")

# Normal Q-Q
stats.probplot(residuals_log, plot=axes[0, 1])
axes[0, 1].set_title("Normal Q-Q (log model)")

# Scale-Location
axes[1, 0].scatter(y_pred_log, np.sqrt(np.abs(residuals_log)), alpha=0.5, color="steelblue")
axes[1, 0].set_title("Scale-Location (log model)")
axes[1, 0].set_xlabel("Fitted Values (log scale)")
axes[1, 0].set_ylabel("sqrt|Residuals|")

# Residuals histogram
axes[1, 1].hist(residuals_log, bins=30, color="steelblue", edgecolor="black")
axes[1, 1].set_title("Distribution of Residuals (log model)")
axes[1, 1].set_xlabel("Residuals")
axes[1, 1].set_ylabel("Frequency")

plt.tight_layout()
plt.show()

In [ ]:
# Compare performance metrics

r2_orig   = r2_score(Y_test, model_lm.predict(X_test))
rmse_orig = root_mean_squared_error(Y_test, model_lm.predict(X_test))

r2_log   = r2_score(Y_test_log, y_pred_log)
rmse_log = root_mean_squared_error(Y_test_log, y_pred_log)        

print(f"Original model  ->  R2: {r2_orig:.4f} | RMSE: {rmse_orig:,.2f}")
print(f"Log model       ->  R2: {r2_log:.4f}  | RMSE: {rmse_log:.4f} (log scale)")

In [ ]:
# Feature engineered model with log transformation and feature selection
base_engine = LinearRegression()

# Forward Stepwise Selector
sfs = SequentialFeatureSelector(
    estimator=base_engine,
    n_features_to_select='auto',
    direction='forward',
    scoring='neg_mean_squared_error',
    cv=5
)

sfs.fit(X_train_log, Y_train_log)

# Get the selected features
best_columns = X_train_log.columns[sfs.get_support()]
print("Features Selected:", list(best_columns))

X_train_best = X_train_log[best_columns]
X_test_best  = X_test_log[best_columns]

# Build and train a new model using selected features
final_log_model = LinearRegression()
final_log_model.fit(X_train_best, Y_train_log)

# Evaluate
y_pred_new = final_log_model.predict(X_test_best)

r2_new   = r2_score(Y_test_log, y_pred_new)
rmse_new = root_mean_squared_error(Y_test_log, y_pred_new)

print(f"New Feature Selected Log Model ->  R2: {r2_new:.4f} | RMSE: {rmse_new:.4f}")

### LOAN DEFAULT CLASSIFICATION MODEL

#### Step 1 — Define features and target

In [ ]:
# Prepare data for classification
dropped_cls_cols = ["loan_status"]

X_cls = df.drop(columns=dropped_cls_cols).copy()
Y_cls = df["loan_status"]


#### Step 2 — Encode education and categoricals

#### Step 3 — Split first, then encode categoricals on train only

In [ ]:
# Split before any fitting to prevent leakage
X_cls_train, X_cls_test, Y_cls_train, Y_cls_test = train_test_split(
    X_cls, Y_cls, test_size=0.3, random_state=42
)

# Fit encoders on train only, transform test
cls_cols_to_encode = ["person_gender", "person_home_ownership", "loan_intent",
                       "previous_loan_defaults_on_file"]

cls_le_encoders = {}
for col in cls_cols_to_encode:
    le = LabelEncoder()
    X_cls_train[col] = le.fit_transform(X_cls_train[col])
    X_cls_test[col]  = le.transform(X_cls_test[col])
    cls_le_encoders[col] = le


#### Step 4 — Check class balance

In [ ]:
# See how split the classes are
sns.countplot(x=Y_cls_train, palette='Set2', edgecolor='black')
plt.title('Class Balance — Train Set')
plt.xticks([0, 1], ['Default', 'Paid'])
plt.ylabel('Count')
plt.show()


#### Step 5 — Handle class imbalance with SMOTE

In [ ]:
# SMOTE synthetically generates new minority class rows so both classes are equal
smote = SMOTE(random_state=42)
X_cls_train_bal, Y_cls_train_bal = smote.fit_resample(X_cls_train, Y_cls_train)

print('Distribution after synthetic generation:', pd.Series(Y_cls_train_bal).value_counts())


#### Step 6 — Baseline: Logistic Regression

In [ ]:
# logistic regression classification
log_cls = LogisticRegression(max_iter=1000, random_state=42)
log_cls.fit(X_cls_train_bal, Y_cls_train_bal)

y_cls_pred_log = log_cls.predict(X_cls_test)
y_cls_prob_log = log_cls.predict_proba(X_cls_test)[:, 1]

print('Logistic Regression')
print(classification_report(Y_cls_test, y_cls_pred_log, target_names=['Default', 'Paid']))
print('ROC-AUC:', round(roc_auc_score(Y_cls_test, y_cls_prob_log), 4))


#### Step 7 — Confusion Matrix: Logistic Regression

In [ ]:
# Confusion matrix shows exactly where the model is making mistakes
cm = confusion_matrix(Y_cls_test, y_cls_pred_log)
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=['Default', 'Paid'], yticklabels=['Default', 'Paid'])
plt.title('Confusion Matrix — Logistic Regression')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()


#### Step 8 — Stronger model: Random Forest

In [ ]:
# Random forest handles non-linear patterns better than logistic regression
rf_cls = RandomForestClassifier(n_estimators=100, random_state=42)
rf_cls.fit(X_cls_train_bal, Y_cls_train_bal)

y_cls_pred_rf = rf_cls.predict(X_cls_test)
y_cls_prob_rf = rf_cls.predict_proba(X_cls_test)[:, 1]

print('Random Forest')
print(classification_report(Y_cls_test, y_cls_pred_rf, target_names=['Default', 'Paid']))
print('ROC-AUC:', round(roc_auc_score(Y_cls_test, y_cls_prob_rf), 4))


#### Step 9 — Confusion Matrix: Random Forest

In [ ]:
# Compare confusion matrix to logistic regression to see if random forest catches more defaults
cm_rf = confusion_matrix(Y_cls_test, y_cls_pred_rf)
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Default', 'Paid'], yticklabels=['Default', 'Paid'])
plt.title('Confusion Matrix — Random Forest')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()


#### Step 10 — ROC Curve: compare both models

In [ ]:
# ROC curve shows the tradeoff between catching defaults vs false alarms at every threshold
fig, ax = plt.subplots(figsize=(8, 6))
RocCurveDisplay.from_predictions(Y_cls_test, y_cls_prob_log, name='Logistic Regression', ax=ax)
RocCurveDisplay.from_predictions(Y_cls_test, y_cls_prob_rf,  name='Random Forest', ax=ax)
ax.plot([0, 1], [0, 1], 'k--', label='Random baseline')
plt.title('ROC Curve Comparison')
plt.legend()
plt.show()


#### Step 11 — Feature Importance (Random Forest)

In [ ]:
# Shows which features the random forest relied on most to predict default
feat_imp = pd.Series(rf_cls.feature_importances_, index=X_cls_train.columns)
feat_imp.sort_values().plot(kind='barh', color='steelblue', edgecolor='black', figsize=(8, 6))
plt.title('Feature Importance — Random Forest')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()
